# hoopstats — fine-tune the ball detector (Colab)

Trains an Ultralytics YOLO model to detect the **basketball** (Phase 2.2 of `doc/plan.md`).

**Before running**

1. Create the dataset on your Mac (`ball_dataset.zip`, YOLO format — see *Dataset format* below) and upload it to
   Google Drive at `MyDrive/hoopstats/ball_dataset.zip`.
2. In Colab: **Runtime → Change runtime type → GPU** (T4 or L4 is enough; A100 is faster but uses more compute units).
3. Run all cells. Checkpoints are written to Drive, so a disconnected session can **resume** by simply running all cells again.
4. When finished: **Runtime → Disconnect and delete runtime** to stop using compute units.

**Dataset format** (inside the zip)
```
ball_dataset/
  data.yaml              # path: ., train: images/train, val: images/val, names: {0: ball}
  images/train/*.jpg     labels/train/*.txt
  images/val/*.jpg       labels/val/*.txt      # one line per ball: 0 cx cy w h (normalised)
```
Images without a ball have an empty label file (these teach the model what is *not* a ball).

In [ ]:
# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Parameters

In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/hoopstats"   # folder on Google Drive
DATASET_ZIP = "ball_dataset.zip"                   # inside DRIVE_DIR
BASE_MODEL  = "yolo11s.pt"      # yolo11n / s / m: bigger = more accurate, slower on the Mac later
IMGSZ       = 1280              # training image size; matches the 1280 px crops the dataset uses
EPOCHS      = 100
PATIENCE    = 25                # early stop if val mAP does not improve for this many epochs
BATCH       = -1                # -1 = pick automatically for the GPU memory
RUN_NAME    = "ball_v1"         # change for each new experiment
ULTRALYTICS = "8.4.163"           # same version as the project's uv.lock

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True)

In [ ]:
# Copy the dataset to the local disk (much faster than reading from Drive during training) and check it
import os, zipfile, glob, yaml, shutil
from pathlib import Path

src = Path(DRIVE_DIR) / DATASET_ZIP
assert src.exists(), f"Upload the dataset first: {src}"
work = Path("/content/data")
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)
with zipfile.ZipFile(src) as z:
    z.extractall(work)
root = next(p.parent for p in work.rglob("data.yaml"))
cfg = yaml.safe_load(open(root / "data.yaml"))
cfg["path"] = str(root)
yaml.safe_dump(cfg, open(root / "data.yaml", "w"))

for split in ("train", "val"):
    imgs = glob.glob(str(root / "images" / split / "*"))
    labels = glob.glob(str(root / "labels" / split / "*.txt"))
    n_balls = sum(1 for f in labels for line in open(f) if line.strip())
    print(f"{split}: {len(imgs)} images, {len(labels)} label files, {n_balls} ball boxes")
print(cfg)

## Train (resumes automatically if a previous run was interrupted)

In [ ]:
from ultralytics import YOLO

project = Path(DRIVE_DIR) / "runs"
last = project / RUN_NAME / "weights" / "last.pt"

if last.exists():
    print("Resuming from", last)
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:          # raised when that run already finished all epochs
        print("Nothing to resume:", e, "\n-> change RUN_NAME to start a new run.")
else:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(root / "data.yaml"),
        imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        project=str(project), name=RUN_NAME, exist_ok=True,
        save_period=5,            # extra checkpoints on Drive
        cache="ram",              # images are small; speeds up epochs
        # small, fast object: keep the ball visible after augmentation
        scale=0.3, degrees=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10,
        plots=True,
    )

## Evaluate

In [ ]:
best = project / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best))
m = model.val(data=str(root / "data.yaml"), imgsz=IMGSZ, conf=0.001, plots=True)
print(f"precision {m.box.mp:.3f}   recall {m.box.mr:.3f}   mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}")

In [ ]:
# Visual check on a few validation images
from IPython.display import Image, display
val_imgs = sorted(glob.glob(str(root / "images" / "val" / "*")))[:6]
for r in model.predict(val_imgs, imgsz=IMGSZ, conf=0.25, save=True, project="/content/pred", name="val", exist_ok=True, verbose=False):
    pass
for f in sorted(glob.glob("/content/pred/val/*"))[:6]:
    display(Image(filename=f, width=640))

## Save the model to Drive

In [ ]:
out = Path(DRIVE_DIR) / "models" / f"{RUN_NAME}.pt"
out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out)
print("Saved", out)
print("On the Mac, copy it to  ~/Code/basketball/data/models/  (the project reads models from there).")
print("Training curves and metrics:", project / RUN_NAME)